In [9]:
# -*-Encoding: utf-8 -*-
import os
import sys
sys.path.insert(0, os.path.abspath('../'))
from data_load.data_loader import Dataset_Custom
from model.model import diffusion_generate, denoise_net, pred_net
from model.jiucai_model import jiucai_net
from torch.optim.lr_scheduler import OneCycleLR, StepLR

from gluonts.torch.util import copy_parameters
from utils.tools import EarlyStopping, adjust_learning_rate
from model.resnet import Res12_Quadratic
from model.diffusion_process import GaussianDiffusion

from model.encoder import Encoder
from model.embedding import DataEmbedding
import numpy as np
import math
import collections
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader
import torch.nn.functional as F

import os
import time
import warnings
import pandas as pd
from utils.timefeatures import time_features
import ast

In [10]:
import argparse
parser = argparse.ArgumentParser(description='generating')

# Load data
parser.add_argument('--root_path', type=str, default='./data/2016', help='root path of the data files')
parser.add_argument('--checkpoints', type=str, default='./checkpoints/', help='location of model checkpoints')
parser.add_argument('--sequence_length', type=int, default=10, help='length of input sequence')
parser.add_argument('--prediction_length', type=int, default=10, help='prediction sequence length')
parser.add_argument('--target_dim', type=int, default=1, help='dimension of target')
parser.add_argument('--input_dim', type=int, default=24, help='dimension of input')
parser.add_argument('--hidden_size', type=int, default=128, help='encoder dimension')
parser.add_argument('--embedding_dimension', type=int, default=64, help='feature embedding dimension')

# Diffusion process
parser.add_argument('--diff_steps', type=int, default=1000, help='number of the diff step')
parser.add_argument('--dropout_rate', type=float, default=0.1, help='dropout')
parser.add_argument('--beta_schedule', type=str, default='linear', help='the schedule of beta')
parser.add_argument('--beta_start', type=float, default=0.0, help='start of the beta')
parser.add_argument('--beta_end', type=float, default=1.0, help='end of the beta')
parser.add_argument('--scale', type=float, default=0.1, help='adjust diffusion scale')

# Bidirectional VAE
parser.add_argument('--arch_instance', type=str, default='res_mbconv', help='path to the architecture instance')
parser.add_argument('--mult', type=float, default=1, help='mult of channels')
parser.add_argument('--num_layers', type=int, default=2, help='num of RNN layers')
parser.add_argument('--num_channels_enc', type=int, default=32, help='number of channels in encoder')
parser.add_argument('--channel_mult', type=int, default=2, help='number of channels in encoder')
parser.add_argument('--num_preprocess_blocks', type=int, default=1, help='number of preprocessing blocks')
parser.add_argument('--num_preprocess_cells', type=int, default=3, help='number of cells per block')
parser.add_argument('--groups_per_scale', type=int, default=2, help='number of cells per block')
parser.add_argument('--num_postprocess_blocks', type=int, default=1, help='number of postprocessing blocks')
parser.add_argument('--num_postprocess_cells', type=int, default=2, help='number of cells per block')
parser.add_argument('--num_channels_dec', type=int, default=32, help='number of channels in decoder')
parser.add_argument('--num_latent_per_group', type=int, default=8, help='number of channels in latent variables per group')

# Training settings
parser.add_argument('--num_workers', type=int, default=1, help='data loader num workers') #set to 1 for debug
parser.add_argument('--patience', type=int, default=3, help='early stopping patience')
parser.add_argument('--itr', type=int, default=5, help='experiment times')
parser.add_argument('--train_epochs', type=int, default=20, help='train epochs')
parser.add_argument('--batch_size', type=int, default=16, help='batch size of train input data')
parser.add_argument('--learning_rate', type=float, default=0.0005, help='optimizer learning rate')
parser.add_argument('--weight_decay', type=float, default=0.0000, help='weight decay')
parser.add_argument('--zeta', type=float, default=0.5, help='trade off parameter zeta')
parser.add_argument('--eta', type=float, default=1.0, help='trade off parameter eta')

parser.add_argument('--use_attn', type=bool, default=True, help='use attention to do cross conditional control')
parser.add_argument('--head_dim', type=int, default=64, help='vetors dim length')
parser.add_argument('--num_heads', type=int, default=2, help='attention head numbers')
parser.add_argument('--query_embedding_dimension', type=int, default=1024, help='embedding llm model dim')


_StoreAction(option_strings=['--query_embedding_dimension'], dest='query_embedding_dimension', nargs=None, const=None, default=1024, type=<class 'int'>, choices=None, required=False, help='embedding llm model dim', metavar=None)

In [11]:
args = parser.parse_args([])

In [12]:
df_test = pd.read_csv('/home/userroot/projs/xiaojiucai/astock/data/test.csv',sep='\t')
df_train = pd.read_csv('/home/userroot/projs/xiaojiucai/astock/data/train.csv',sep='\t')

In [13]:
df_news = pd.read_parquet('/home/userroot/projs/xiaojiucai/news_data/news.parquet')

In [14]:
df_kline = pd.read_parquet('/home/userroot/projs/xiaojiucai/news_data/kline.parquet')

In [15]:
tmp = pd.Series([1,2,3])
tmp.index.values[0]

0

In [16]:
df_kline[df_kline['收盘'] == 0].sort_index()

,股票名称,股票代码,日期,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率
0,伟星新材,002372,2010-03-18,-0.29,0.0,0.00,-0.29,401304,9.561472e+08,-24.37,100.0,1.19,79.12
1,华峰化学,002064,2006-08-24,-0.04,0.0,0.00,-0.05,110143,1.003714e+08,-500.00,100.0,0.01,38.78
2,海立股份,600619,1992-11-18,-0.25,0.0,0.02,-0.29,12367,7.356349e+06,-119.23,100.0,0.26,154.59
3,兴业银行,601166,2007-02-08,0.09,0.0,0.13,-0.16,341531,8.311912e+08,170.59,-100.0,-0.17,7.58
5,光大嘉宝,600622,1992-12-10,0.07,0.0,0.16,-0.03,6805,1.071500e+07,475.00,-100.0,-0.04,8.51
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,福耀玻璃,600660,2014-04-04,-0.03,0.0,0.06,-0.08,98084,8.211491e+07,0.00,0.0,0.00,0.49
5003,福耀玻璃,600660,2014-04-17,0.08,0.0,0.11,0.00,44664,3.771743e+07,0.00,0.0,0.00,0.22
5117,万 科Ａ,000002,2012-03-09,0.05,0.0,0.10,-0.10,362592,3.110829e+08,2000.00,-100.0,-0.01,0.38
5145,万 科Ａ,000002,2012-04-23,0.02,0.0,0.05,-0.10,351764,3.013286e+08,375.00,-100.0,-0.04,0.36


In [17]:
import scipy.stats as stats
df_kline.columns

Index(['股票名称', '股票代码', '日期', '开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌幅',
       '涨跌额', '换手率'],
      dtype='object')

In [29]:
def move_percentage(x):

    x1, x2 = x
    if x1 == 0:
        return np.NaN
    ret = (x2-x1)/x1
    return ret
count = 0

for name , group in df_kline.groupby('股票代码'):
    if count >= 200:
        break
    count += 1
    index = group['日期'] >= '2023-05-27'
    
    code = (group['股票代码'][index]).values
    date = (group['日期'][index]).values
    s_size = date.size

    # 如果这个股票交易天数不足，剔除掉
    if s_size < 100:
        continue
    
    # 因为rolling操作 第一个是nan 所以做了 对齐的工作
    close_movement = (group['收盘'][index].rolling(2).apply(move_percentage).fillna(method='bfill').fillna(method='ffill')).values
    # if close_movement.size > s_size:
    #     close_movement = close_movement[1:]
    # else:
    #     close_movement[0] = 0
    open_zscore = stats.zscore(group['开盘'][index].values)
    close_zscore = stats.zscore(group['收盘'][index].values)
    high_zscore = stats.zscore(group['最高'][index].values)
    low_zscore = stats.zscore(group['最低'][index].values)
    trade_volumne_zscore = stats.zscore(group['成交量'][index].values)
    biz_volumne_zscore = stats.zscore(group['成交额'][index].values)
    #print(f'close movement size is {close_movement.size}')
    #print(f'code size is {code.size}')
    pd.DataFrame(np.vstack([date, code, close_movement, open_zscore, close_zscore, high_zscore, low_zscore, trade_volumne_zscore, biz_volumne_zscore]).transpose()).to_parquet(f'./data_out/{name}.parquet')
    

In [54]:
for name , group in df_kline.groupby('股票代码'):
    if count >= 200:
        break
    code = (group['股票代码']).values
    date = (group['日期']).values
    index = group['日期'] > '2023-05-25'
    print(group['日期'][index])
    break

7676    2023-05-26
7677    2023-05-29
7678    2023-05-30
7679    2023-05-31
7680    2023-06-01
           ...    
7873    2024-03-19
7874    2024-03-20
7875    2024-03-21
7876    2024-03-22
7877    2024-03-25
Name: 日期, Length: 202, dtype: object


In [53]:
index

0       False
1       False
2       False
3       False
4       False
        ...  
7873     True
7874     True
7875     True
7876     True
7877     True
Name: 日期, Length: 7878, dtype: bool

In [48]:
df_news = pd.read_parquet("/home/userroot/projs/xiaojiucai/news_data/news.parquet")

In [7]:
import pandas as pd
df_kline = pd.read_parquet("/home/userroot/projs/xiaojiucai/xiaojiucai_data/kline/000001.parquet")
df_kline = df_kline.rename(columns={0:"date", 1:"code"})  
                           #columns=["date", "code", "close_movement", "open_zscore", "close_zscore", "high_zscore", "low_zscore", "trade_volumne_zscore", "biz_volumne_zscore"])
df_kline

,date,code,2,3,4,5,6,7,8
0,2023-05-26,000001,NaN,1.110707,1.306391,1.218287,1.113590,-0.789411,-0.688416
1,2023-05-29,000001,-0.010152,1.306079,1.175031,1.218287,1.233730,-0.709579,-0.595957
2,2023-05-30,000001,-0.009402,1.164977,1.054618,1.067084,1.069903,-0.420428,-0.281694
3,2023-05-31,000001,-0.023296,0.991313,0.759059,0.905080,0.829624,0.186720,0.372359
4,2023-06-01,000001,-0.000883,0.752525,0.748112,0.732277,0.742249,-0.451069,-0.343798
...,...,...,...,...,...,...,...,...,...
197,2024-03-19,000001,-0.013283,-0.104940,-0.248034,-0.196543,-0.164259,0.352723,0.336644
198,2024-03-20,000001,0.004808,-0.267750,-0.193301,-0.272144,-0.186103,-0.261674,-0.276185
199,2024-03-21,000001,0.001914,-0.191772,-0.171407,-0.218143,-0.131494,-0.273650,-0.282937
200,2024-03-22,000001,-0.010506,-0.191772,-0.291821,-0.293745,-0.251634,-0.034531,-0.060332


In [35]:
np.vstack([open_zscore,high_zscore]).transpose()

array([[-1.11744931, -1.11967257],
       [-1.11744931, -1.11967257],
       [-1.11744931, -1.11967257],
       ...,
       [ 1.03827696,  1.01774535],
       [ 1.03827696,  1.00503343],
       [ 1.01985195,  1.01229738]])

In [5]:
stock = df_train[['CODE', 'DATE', 'text_a', 'stock_factors']]
sorted_stock = stock.sort_values(['CODE', 'DATE'])
sorted_stock = sorted_stock.reset_index(drop=True)

In [6]:
df_stamp = pd.DatetimeIndex(sorted_stock['DATE']) #训练数据集的日期标签
data_stamp = time_features(df_stamp) #把日期时间序列转化为多个时间层次的序列，并且把序列的数值归一到【-0.5， +0.5】

# output, y_noisy, dsm_loss = self.denoise_net(batch_x, x_mark, batch_y, t)

In [7]:
sorted_stock['stock_factors'] = sorted_stock['stock_factors'].apply(ast.literal_eval)

In [18]:
xx = []
X = None
tt = []
cnt = 0
T = None
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
emb_model = HuggingFaceEmbedding(model_name='/data/models/bge-large-zh')
for g in sorted_stock.groupby('CODE'):
    #print(len(g[1]))
    
    
    if len(g[1]) > 10:
        tmp = g[1]['stock_factors'].iloc[0:args.sequence_length].apply(np.array)
        txt_list = g[1]['text_a'].iloc[0:args.sequence_length].apply(emb_model.get_text_embedding)
        x = torch.tensor(np.array(tmp.values.tolist()), dtype=torch.float32)
        t = torch.tensor(np.array(txt_list.values.tolist()), dtype=torch.float32)
        #x = x.squeeze()
        #print(x.shape)
        xx.append(x)
        tt.append(t)
        cnt += 1
        #break
    if cnt == args.batch_size:
        X = torch.stack(xx)
        T = torch.stack(tt)
        break

print(X.shape)
print(T.shape)

torch.Size([16, 10, 24])
torch.Size([16, 10, 1024])


In [19]:
tmp = sorted_stock[['stock_factors']].iloc[0:args.sequence_length].apply(np.array)
x_input = X.to('cuda')
text_emb_input = T.to('cuda')
y = x_input
t = torch.tensor(data_stamp[0:args.sequence_length])
t = t.unsqueeze(0).to('cuda')
diff_t = torch.randint(0, args.diff_steps, (args.batch_size,)).long().to('cuda')

In [12]:
x_input.shape

torch.Size([16, 10, 24])

In [20]:
%load_ext autoreload

%autoreload 2
%reload_ext autoreload
from model.jiucai_model import jiucai_net
#import importlib
#importlib.reload(jiucai_net)
d_net = denoise_net(args).to('cuda')
j_net = jiucai_net(args).to('cuda')

In [21]:
%reload_ext autoreload
j_net(x_input,t,y,diff_t)

attention doing


(<model.encoder.NormalDecoder at 0x7f65f8e3aad0>,
 tensor([[[[ 0.0705,  0.4141, -1.6570,  ..., -1.7428,  1.0855,  0.9051],
           [ 1.2695,  0.2520,  0.4438,  ..., -0.3076,  0.8752, -0.5215],
           [ 0.7353,  0.9192, -0.2130,  ...,  1.4409,  0.4951,  0.3614],
           ...,
           [ 1.2943,  1.8071,  0.8043,  ...,  0.6037,  0.6523, -2.0040],
           [-1.1114,  0.4603, -1.3795,  ..., -1.2684,  0.1397,  1.2119],
           [ 0.6983, -0.7983,  0.4079,  ...,  0.9793,  2.1271,  0.2721]]],
 
 
         [[[-0.1145, -0.2839,  1.5800,  ...,  0.7973, -0.8401,  2.1740],
           [ 1.3571, -0.7943, -2.0101,  ...,  0.1304, -0.2199, -0.9139],
           [ 0.8262,  0.1602, -0.4249,  ...,  0.9240,  0.7119, -2.2541],
           ...,
           [-0.0613,  0.5978, -1.9681,  ...,  0.9266, -0.0557,  1.3073],
           [ 1.2382,  0.3256, -0.1455,  ...,  1.4947, -0.4739,  0.4685],
           [-0.1674,  0.5758,  0.0747,  ..., -0.8321, -2.6555,  1.8373]]],
 
 
         [[[-0.7612,  1.3932, 

In [18]:
d_net(x_input,t,y,diff_t)

(<model.encoder.NormalDecoder at 0x7f107c1974f0>,
 tensor([[[[ 0.3403,  2.6290, -0.5521,  ..., -0.4688,  0.5646,  1.3136],
           [ 0.1651, -0.6179,  0.7537,  ...,  0.6219,  1.0547, -0.5237],
           [-0.2875,  0.8194, -0.8593,  ...,  0.0109, -0.0260, -2.3500],
           ...,
           [-0.5739,  0.3590,  1.1618,  ...,  0.0195, -0.6214,  2.0949],
           [ 1.4808,  0.1285,  0.3615,  ..., -2.1650, -1.3665,  0.2000],
           [-0.0446,  0.4247, -1.7133,  ..., -0.5341,  0.6213, -0.3703]]],
 
 
         [[[ 0.2550,  0.1486,  0.6640,  ..., -0.7608, -0.0219,  0.4251],
           [-0.9666,  0.3326, -0.3552,  ..., -1.1109,  1.6931, -0.1372],
           [ 0.9941,  0.5534,  0.2706,  ..., -0.7913, -0.4540, -1.0752],
           ...,
           [-0.1274,  0.8503, -0.7400,  ..., -0.9243, -0.4481,  1.7436],
           [-0.2217, -0.2210, -1.0468,  ...,  0.4021,  0.7329, -0.7587],
           [ 0.7906,  0.3310, -0.9997,  ...,  0.4705,  0.3664,  0.6591]]],
 
 
         [[[ 0.1609, -0.4808, 